# The oracle control at a full training budget (O2)

Registered in `docs/protocol.md` ("Registered before running: the oracle control at a full training budget
(O2)"). Perfect selection plus 20 surface forms per fact, three epochs, three seeds, on the original
synthetic corpus. Valid only if closed-book accuracy reaches 0.5.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')



## Corpus, compaction, references


In [ ]:
import os, sys, json, shutil
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_synthetic.yaml'
COMPACTOR = 'Qwen/Qwen2.5-1.5B-Instruct'
TARGET = 'Qwen/Qwen2.5-0.5B-Instruct'
SEEDS = [0, 1, 2]
OUT = '/kaggle/working/o2'
os.makedirs(OUT, exist_ok=True)

from common.io import load_config
_c = load_config(CFG)
RUNS, DATA = _c['run_root'], _c['data']['dir']
CASC = f'{RUNS}/cascading'
NONE = f'{RUNS}/none_context'
run('python data/generate_synthetic.py --n-train 48 --n-eval 48 --n-turns 120')
run(f'python baselines/cascading.py --config {CFG} --split both --out {CASC} --set model.base={COMPACTOR}')
run(f'python eval/span_report.py --events {CASC}/eval_events.jsonl --show 0 --out {OUT}/span_report.json')
run(f'python baselines/full_context.py --config {CFG} --split eval --mode none --out {NONE}')
run(f'python eval/retention.py --config {CFG} --contexts {CASC}/eval_contexts.jsonl --label cascading '
    f'--out {OUT}/cascading --set model.base={TARGET}')


## Oracle seeds


In [ ]:
for seed in SEEDS:
    orc = f'{RUNS}/oracle_s{seed}'
    dst = f'{OUT}/seed_{seed}'
    os.makedirs(dst, exist_ok=True)
    run(f'python sleep/oracle.py --config {CFG} --trajectories {DATA}/eval.jsonl --run-dir {orc} --n-forms 20 '
        f'--set model.base={TARGET} seed={seed}')
    for f in ('result.json',):
        if os.path.exists(f'{orc}/{f}'):
            shutil.copy(f'{orc}/{f}', f'{dst}/train_{f}')
    for label, ctx in (('oracle-closedbook', f'{NONE}/eval_contexts.jsonl'), ('oracle', f'{CASC}/eval_contexts.jsonl')):
        run(f'python eval/retention.py --config {CFG} --contexts {ctx} --label {label} --out {dst}/{label} '
            f'--adapter {orc}/adapter --set model.base={TARGET} seed={seed}')
    shutil.make_archive('/kaggle/working/o2_runs', 'zip', OUT)
    r = {l: json.load(open(f'{dst}/{l}.summary.json')) for l in ('oracle-closedbook', 'oracle')}
    print(f"seed {seed}: closed book {r['oracle-closedbook']['retention_accuracy']:.3f}  "
          f"with context: all {r['oracle']['retention_accuracy']:.3f} evicted {r['oracle']['evicted_accuracy']:.3f}", flush=True)


## Registered readout


In [ ]:
UNIFORM_RECORDED = 0.123
cb = [json.load(open(f'{OUT}/seed_{s}/oracle-closedbook.summary.json'))['retention_accuracy'] for s in SEEDS]
ev = [json.load(open(f'{OUT}/seed_{s}/oracle.summary.json'))['evicted_accuracy'] for s in SEEDS]
casc = json.load(open(f'{OUT}/cascading.summary.json'))
valid = sum(cb) / len(cb) >= 0.5
res = {'closed_book': cb, 'closed_book_mean': sum(cb) / len(cb), 'evicted_with_context': ev,
       'evicted_with_context_mean': sum(ev) / len(ev), 'cascading_evicted': casc['evicted_accuracy'],
       'cascading_all': casc['retention_accuracy'], 'valid': valid,
       'headroom_real': bool(valid and all(e > UNIFORM_RECORDED for e in ev)),
       'verdict': ('valid control' if valid else 'control failed again: uninformative')}
json.dump(res, open(f'{OUT}/readout.json', 'w'), indent=1)
shutil.make_archive('/kaggle/working/o2_runs', 'zip', OUT)
print(json.dumps(res, indent=1))
